In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Configuration
BASE_DIR = Path.cwd()
for candidate in [BASE_DIR, *BASE_DIR.parents]:
    if (candidate / "preprocessing_images").exists() and (candidate / "Lung" / "inference_results").exists():
        BASE_DIR = candidate
        break
PREPROCESSING_IMAGES_DIR = BASE_DIR / "preprocessing_images"
INFERENCE_DIR = BASE_DIR / "Lung" / "inference_results"

# 1. Discover all unique image IDs
all_files = list(PREPROCESSING_IMAGES_DIR.glob("*.png"))
image_ids = sorted(list(set(f.name.split('_')[0] for f in all_files)))
print(f"Found {len(image_ids)} unique image IDs.")

# 2. SELECT IMAGE INDEX HERE (1 to 13)
IMAGE_INDEX = 3  # <-- Change this to pick a different image (1 to 13)

if not (1 <= IMAGE_INDEX <= len(image_ids)):
    print(f"Error: Index must be between 1 and {len(image_ids)}")
else:
    image_id = image_ids[IMAGE_INDEX - 1]
    print(f"### DISPLAYING RESULTS FOR IMAGE ID: {image_id} ###\n")

    def visualize_horizontal(image_list, title_prefix=""):
        cols = len(image_list)
        # figsize=(Rộng, Cao)
        fig, axes = plt.subplots(1, cols, figsize=(28, 8))
        if cols == 1: axes = [axes]
        
        for i, img_info in enumerate(image_list):
            if img_info['path'].exists():
                img = cv2.imread(str(img_info['path']))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[i].imshow(img)
                
                # CHỈNH CỠ CHỮ TIÊU ĐỀ ẢNH TẠI ĐÂY (fontsize)
                axes[i].set_title(img_info['title'], fontsize=25, fontweight='bold') 
            else:
                axes[i].text(0.5, 0.5, f"File Not Found:\n{img_info['path'].name}", 
                             ha='center', va='center', color='red', fontsize=18)
            axes[i].axis('off')
        
        # CHỈNH CỠ CHỮ TIÊU ĐỀ TỔNG TẠI ĐÂY (fontsize)
        plt.suptitle(title_prefix, fontsize=30, fontweight='bold', y=1.05)
        
        plt.tight_layout()
        fig.subplots_adjust(wspace=0.03)
        plt.show()

    # --- PLOT 1: INFERENCE RESULTS (1x4) ---
    plot1_methods = [
        {"path": BASE_DIR / f"{image_id}_dicom.png", "title": "Original DICOM"},
        {"path": BASE_DIR / f"{image_id}_groundtruth.png", "title": "Ground Truth"},
        {"path": INFERENCE_DIR / image_id / f"pred_{image_id}_raw_minmax.png", "title": "Raw Min-Max Prediction"},
        {"path": INFERENCE_DIR / image_id / f"pred_{image_id}_raw_plus_voi_plus_percentile.png", "title": "Raw+VOI+Percentile Prediction"}
    ]
    visualize_horizontal(plot1_methods, title_prefix=f"Inference Results: {image_id}")

    # --- PLOT 2: PREPROCESSED BACKGROUNDS (1x4) ---
    plot2_suffixes = [
        ("voi_plus_percentile_plus_clahe", "VOI LUT + Percentile + CLAHE"),
        ("raw_plus_voi_plus_percentile", "Raw Min-Max + VOI LUT + Percentile"),
        ("raw_plus_voi_plus_clahe", "Raw Min-Max + VOI LUT + CLAHE"),
        ("raw_plus_percentile_plus_clahe", "Raw Min-Max + Percentile + CLAHE")
    ]
    
    plot2_images = []
    for suffix, title in plot2_suffixes:
        path = PREPROCESSING_IMAGES_DIR / f"{image_id}_{suffix}.png"
        plot2_images.append({"path": path, "title": title})
    
    visualize_horizontal(plot2_images)

    print(f"\\nFinal Display Completed for Image: {image_id}")